In [1]:
pip install webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pandas


Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [4]:
from selenium import webdriver
from selenium. webdriver.chrome.options import Options
from selenium. webdriver.common.by import By
import pandas as pd
import time
import re

In [5]:
# ==========================
# CONFIGURACIÓN
# ==========================
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-notifications")

driver = webdriver.Chrome(options=options)

url = "https://www.elespectador.com/buscador/migraci%C3%B3n-venezolana/"
driver.get(url)

print("⏳ Esperando carga inicial...")
time.sleep(10)

⏳ Esperando carga inicial...


In [ ]:
# ==========================
# SCROLL MODERADO
# ==========================
print("\n📜 Haciendo scroll...")

for i in range(10):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(4)
    print(f"   Scroll {i+1}/10")

print("\n" + "="*60)
print("🔍 INICIANDO EXTRACCIÓN CON DEBUGGING")
print("="*60)

# up to this point everthing is good.


📜 Haciendo scroll...
   Scroll 1/10
   Scroll 2/10
   Scroll 3/10
   Scroll 4/10
   Scroll 5/10
   Scroll 6/10
   Scroll 7/10
   Scroll 8/10
   Scroll 9/10
   Scroll 10/10

🔍 INICIANDO EXTRACCIÓN CON DEBUGGING


In [7]:
# ==========================
# DEBUGGING PASO A PASO
# ==========================

# PASO 1: Ver todos los H2 y H3
print("\n📌 PASO 1: Buscando todos los H2 y H3...")
h2_todos = driver.find_elements(By.TAG_NAME, 'h2')
h3_todos = driver.find_elements(By.TAG_NAME, 'h3')
print(f"   ✓ H2 encontrados: {len(h2_todos)}")
print(f"   ✓ H3 encontrados: {len(h3_todos)}")

# PASO 2: Ver H2/H3 con enlaces dentro
print("\n📌 PASO 2: Buscando H2 y H3 con enlaces <a>...")
h2_con_enlaces = driver.find_elements(By.XPATH, '//h2/a')
h3_con_enlaces = driver.find_elements(By.XPATH, '//h3/a')
print(f"   ✓ H2 con <a>: {len(h2_con_enlaces)}")
print(f"   ✓ H3 con <a>: {len(h3_con_enlaces)}")

# Mostrar ejemplos
if h2_con_enlaces: 
    print(f"\n   Ejemplo H2:")
    for i, elem in enumerate(h2_con_enlaces[: 3]):
        print(f"      {i+1}.  Texto: '{elem.text[: 60]}...'")
        print(f"         URL: {elem.get_attribute('href')}")

if h3_con_enlaces:
    print(f"\n   Ejemplo H3:")
    for i, elem in enumerate(h3_con_enlaces[:3]):
        print(f"      {i+1}.  Texto: '{elem.text[: 60]}...'")
        print(f"         URL: {elem.get_attribute('href')}")

# PASO 3: Buscar por IDs específicos
print("\n📌 PASO 3: Buscando en bloques específicos...")
bloques_ids = [
    'main-layout-2',
    'main-layout-6-7',
    'home_bloque_Reportajes'
]

elementos_por_bloque = []
for bloque_id in bloques_ids: 
    try:
        # Buscar el bloque
        bloque = driver.find_elements(By.ID, bloque_id)
        if bloque:
            # Buscar enlaces dentro del bloque
            enlaces_h2 = driver.find_elements(By. XPATH, f'//*[@id="{bloque_id}"]//h2/a')
            enlaces_h3 = driver.find_elements(By.XPATH, f'//*[@id="{bloque_id}"]//h3/a')
            total = len(enlaces_h2) + len(enlaces_h3)
            print(f"   ✓ {bloque_id}: {total} enlaces (H2: {len(enlaces_h2)}, H3: {len(enlaces_h3)})")
            elementos_por_bloque.extend(enlaces_h2)
            elementos_por_bloque.extend(enlaces_h3)
        else:
            print(f"   ✗ {bloque_id}: NO ENCONTRADO")
    except Exception as e:
        print(f"   ✗ {bloque_id}: ERROR - {e}")

# PASO 4: Buscar TODOS los enlaces que contengan "politica"
print("\n📌 PASO 4: Todos los enlaces que contengan 'politica'...")
todos_enlaces_politica = driver.find_elements(By. XPATH, '//a[contains(@href, "/politica/")]')
print(f"   ✓ Enlaces con '/politica/': {len(todos_enlaces_politica)}")

# Mostrar algunos ejemplos
if todos_enlaces_politica:
    print(f"\n   Primeros 5 enlaces con '/politica/':")
    for i, elem in enumerate(todos_enlaces_politica[:5]):
        texto = elem.text. strip()
        url = elem.get_attribute('href')
        print(f"      {i+1}. '{texto[: 50]}...' -> {url}")


📌 PASO 1: Buscando todos los H2 y H3...
   ✓ H2 encontrados: 10
   ✓ H3 encontrados: 21

📌 PASO 2: Buscando H2 y H3 con enlaces <a>...
   ✓ H2 con <a>: 10
   ✓ H3 con <a>: 10

   Ejemplo H2:
      1.  Texto: 'Migrantes en Bello: así deben actualizar sus datos para no p...'
         URL: https://www.elespectador.com/mundo/venezuela/migrantes-en-bello-antioquia-asi-deben-actualizar-sus-datos-para-no-perder-el-servicio-de-salud/
      2.  Texto: 'Petro en EE. UU.: “Hay que avanzar más” en los derechos de l...'
         URL: https://www.elespectador.com/mundo/america/petro-en-ee-uu-hay-que-avanzar-mas-en-los-derechos-de-los-migrantes-venezolanos/
      3.  Texto: '“Emigro, luego existo”: la diáspora venezolana narrada desde...'
         URL: https://www.elespectador.com/el-magazin-cultural/emigro-luego-existo-la-diaspora-venezolana-narrada-desde-el-grafiti/

   Ejemplo H3:
      1.  Texto: 'Redacción Internacional...'
         URL: None
      2.  Texto: 'Redacción Internacional...'
      

In [8]:
# ==========================
# EXTRACCIÓN SIN FILTROS ESTRICTOS
# ==========================
print("\n" + "="*60)
print("📊 EXTRAYENDO DATOS (sin filtros estrictos)")
print("="*60)

# Combinar todas las fuentes
todos_elementos = list(set(h2_con_enlaces + h3_con_enlaces + elementos_por_bloque))

titulares = []
links = []
categorias = []
enlaces_unicos = set()

contador_descartados = {
    'sin_texto': 0,
    'sin_url': 0,
    'url_invalida': 0,
    'duplicado': 0,
    'muy_corto': 0
}

for elem in todos_elementos:
    try:
        titulo = elem.text.strip()
        enlace = elem.get_attribute("href")
        
        # Debug:  contar por qué se descartan
        if not titulo:
            contador_descartados['sin_texto'] += 1
            continue
        
        if not enlace:
            contador_descartados['sin_url'] += 1
            continue
        
        if "elespectador.com" not in enlace:
            contador_descartados['url_invalida'] += 1
            continue
        
        if enlace in enlaces_unicos: 
            contador_descartados['duplicado'] += 1
            continue
        
        if len(titulo) < 10:
            contador_descartados['muy_corto'] += 1
            continue
        
        # Detectar categoría
        match = re.search(r"elespectador\.com/([^/]+)/", enlace)
        categoria = match.group(1).capitalize() if match else "General"
        
        # AGREGAR (filtros mínimos)
        titulares.append(titulo)
        links.append(enlace)
        categorias.append(categoria)
        enlaces_unicos. add(enlace)
        
    except Exception as e:
        print(f"   ⚠️ Error procesando elemento: {e}")
        continue

# Mostrar estadísticas de descarte
print("\n📉 Elementos descartados:")
for motivo, cantidad in contador_descartados.items():
    print(f"   - {motivo}: {cantidad}")

print(f"\n✅ Elementos ACEPTADOS: {len(titulares)}")



📊 EXTRAYENDO DATOS (sin filtros estrictos)

📉 Elementos descartados:
   - sin_texto: 0
   - sin_url: 10
   - url_invalida: 0
   - duplicado: 0
   - muy_corto: 0

✅ Elementos ACEPTADOS: 10


In [9]:
# ==========================
# GUARDAR
# ==========================
if len(titulares) > 0:
    df = pd.DataFrame({
        "Titular": titulares,
        "Categoría": categorias,
        "Link": links
    })
    
    df.to_csv("titulares_elespectador_DEBUG.csv", index=False, encoding="utf-8-sig")
    
    print(f"\n📊 RESULTADOS FINALES:")
    print(f"   Total:  {len(df)} noticias")
    print(f"\n📰 Primeras 10 noticias:")
    for idx, row in df.head(10).iterrows():
        print(f"   {idx+1}. {row['Titular'][: 70]}...")
    
    print(f"\n📁 Archivo:  titulares_elespectador_DEBUG.csv")
else:
    print("\n❌ NO SE ENCONTRARON TITULARES")
    print("\n🔍 Intentando estrategia alternativa:  capturar TODO...")
    
    # PLAN B: Capturar ABSOLUTAMENTE TODO
    todos_los_a = driver.find_elements(By. TAG_NAME, 'a')
    print(f"\n   Total de enlaces <a> en la página: {len(todos_los_a)}")
    
    titulares_plan_b = []
    links_plan_b = []
    
    for a in todos_los_a: 
        try:
            texto = a.text.strip()
            url = a.get_attribute('href')
            
            if (texto and url and 
                "elespectador.com" in url and 
                len(texto) > 20 and
                "/politica/" in url):
                
                titulares_plan_b.append(texto)
                links_plan_b.append(url)
        except: 
            continue
    
    print(f"\n   ✓ Con PLAN B encontrados: {len(titulares_plan_b)}")
    
    if titulares_plan_b: 
        df_plan_b = pd.DataFrame({
            "Titular": titulares_plan_b,
            "Link": links_plan_b
        })
        df_plan_b.to_csv("titulares_elespectador_PLANB.csv", index=False, encoding="utf-8-sig")
        print(f"   📁 Guardado en: titulares_elespectador_PLANB. csv")

driver.quit()
print("\n✅ Proceso finalizado")


📊 RESULTADOS FINALES:
   Total:  10 noticias

📰 Primeras 10 noticias:
   1. Gobierno alista cirugía a solicitudes de asilo: hay más de 30 mil proc...
   2. Trump, Bukele y migrantes venezolanos: una combinación explosiva...
   3. A orillas de Venezuela, los autores que viven en el exilio...
   4. Migrantes en Bello: así deben actualizar sus datos para no perder el s...
   5. El desafío migratorio no cesa en Maicao, La Guajira...
   6. La migración venezolana cambia de ruta...
   7. Los cambios en la migración de venezolanos a Cúcuta...
   8. “Hemos llegado a Berlín”: la migración venezolana vista desde la infan...
   9. Petro en EE. UU.: “Hay que avanzar más” en los derechos de los migrant...
   10. “Emigro, luego existo”: la diáspora venezolana narrada desde el grafit...

📁 Archivo:  titulares_elespectador_DEBUG.csv

✅ Proceso finalizado


In [10]:
import os
print(os.getcwd())

c:\Users\diego\ScrappingPI2
